Transfer Learning – 3-Class Image Classification

- ResNet → residual connections (very stable training)
- EfficientNet → compound scaling (depth/width/resolution)
- MobileNet → lightweight (depthwise separable conv)
- DenseNet → feature reuse (concat instead of add)
- ViT → patch + attention instead of CNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

# -------------------------
# Data
# -------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder("train/", transform=transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

# -------------------------
# Model (Transfer Learning)
# -------------------------
import torch.nn as nn
from torchvision import models
import timm


# -----------------------------
# Backbone Freezing # Freeze all layers except the head
# -----------------------------
def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    return model


# -----------------------------
# Unified Model Builder for Multiple Architectures
# -----------------------------
def get_model(name="resnet18", num_classes=3, pretrained=True, freeze=True):

    # =========================
    # 1. ResNet (fc head)
    # =========================
    if name == "resnet18":
        model = models.resnet18(pretrained=pretrained)
        if freeze:
            model = freeze_backbone(model)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if name == "resnet50":
        model = models.resnet50(pretrained=pretrained)
        if freeze:
            model = freeze_backbone(model)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model


    # =========================
    # 2. EfficientNet (classifier head)
    # =========================
    if name == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=pretrained)
        if freeze:
            model = freeze_backbone(model)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        return model


    # =========================
    # 3. MobileNetV3 (classifier head)
    # =========================
    if name == "mobilenetv3":
        model = models.mobilenet_v3_large(pretrained=pretrained)
        if freeze:
            model = freeze_backbone(model)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
        return model


    # =========================
    # 4. DenseNet (classifier head)
    # =========================
    if name == "densenet":
        model = models.densenet121(pretrained=pretrained)
        if freeze:
            model = freeze_backbone(model)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
        return model


    # =========================
    # 5. Vision Transformer (ViT head)
    # =========================
    if name == "vit":
        model = timm.create_model("vit_base_patch16_224", pretrained=pretrained)
        if freeze:
            model = freeze_backbone(model)
        model.head = nn.Linear(model.head.in_features, num_classes)
        return model


    raise ValueError(f"Unknown model name: {name}")

model = get_model(name="resnet18", num_classes=3, pretrained=True, freeze=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# -------------------------
# Loss & Optimizer
# -------------------------
criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.fc.parameters(), lr=1e-3) # good for resnet and classifier heads
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

# -------------------------
# Training Loop
# -------------------------
for epoch in range(5):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)  # [B, 3]
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item()}")

# -------------------------
# Evaluation Loop
# -------------------------
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for x, y in train_loader: # replace by test_loader for evaluation on test set
        x, y = x.to(device), y.to(device)

        logits = model(x)  # [B, 3]
        preds = torch.argmax(logits, dim=1)  # [B]

        correct += (preds == y).sum().item()
        total += y.size(0)

Segmentation – U-Net (3 Downsampling Blocks)
Task: pixel-wise classification (3 classes)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------
# Simple U-Net Block
# -------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)

# -------------------------
# U-Net (3 levels)
# -------------------------
class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=3):
        super().__init__()

        self.enc1 = ConvBlock(in_channels, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = ConvBlock(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ConvBlock(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ConvBlock(128, 64)

        self.out = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)  # [B, 3, H, W]


criterion = nn.CrossEntropyLoss()  # expects [B, C, H, W]
model = UNet(in_channels=3, num_classes=3).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        images, labels = images.to(device), labels.to(device)
        # images: [B, 3, H, W]
        # labels: [B, H, W] (each pixel has a class label in [0, num_classes-1])
        logits = model(images)  # [B, 3, H, W]

        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



Object Detection

In [ ]:
## Faster R-CNN (PyTorch built-in)

import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

# Replace classifier head
num_classes = 4  # 3 classes + background
in_features = model.roi_heads.box_predictor.cls_score.in_features # get the number of input features for the classifier

model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes) # replace the head with a new one 

model = model.to("cuda")

# Loss & Optimizer
criterion = nn.CrossEntropyLoss() # for classification head
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

# Training Loop

model.train()

for images, targets in dataloader:
    images = [img.to(device) for img in images]
    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

    # Forward pass (returns losses in training mode)
    loss_dict = model(images, targets)

    loss = sum(loss for loss in loss_dict.values())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(loss.item())


## YOLO
# Dataset Structure:
# YOLO expects images and labels in separate train/validation folders, where each image has a matching .txt annotation file.

# Label Format:
# Each label file contains one object per line as: class_id x_center y_center width height, with all coordinates normalized to [0,1].

# data.yaml:
# Defines the dataset paths, number of classes (nc), and class names used during training and validation.

from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # pretrained

# Train
model.train(
    data="data.yaml",
    epochs=10,
    imgsz=640
)

# Validation
metrics = model.val()

print(metrics.box.map50)   # mAP@0.5
print(metrics.box.map)     # mAP@0.5:0.95

# Inference
results = model("test.jpg")
for r in results:
    r.show()   # opens image with boxes

# Get Predictions Programmatically
for r in results:
    boxes = r.boxes

    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        conf = box.conf[0]
        cls = box.cls[0]

        print(f"Class: {cls}, Confidence: {conf}")

Vision Transformers (ViT)

In [ ]:
import timm
import torch
import torch.nn as nn

model = timm.create_model("vit_base_patch16_224", pretrained=True)

# Replace classifier head
model.head = nn.Linear(model.head.in_features, 3)

model = model.to("cuda")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
model.train()
for images, labels in train_loader:
    images, labels = images.cuda(), labels.cuda()

    logits = model(images)
    loss = criterion(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.cuda(), labels.cuda()

        logits = model(images)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Accuracy: {correct / total}")